# CS781 Phase 3: ModernBERT-style encoder (written for architecture coverage)

> **STATUS: NOT RUN. This notebook has no leaderboard score and no results.** It applies the Phase 1 ModernBERT-style encoder to Phase 3, completing the architecture coverage of the repository. It reuses the cells of the Phase 3 pipeline that ran on Kaggle (sentence classification into 13 rhetorical roles for the CL and IT domains, with context sentences) and swaps only the backbone: one experiment, full fine-tuning, mean pooling over the tokens for the non-causal model.
>
> **What was checked:** all cells were compiled; the backbone registration and the classifier forward pass were checked on CPU with a tiny randomly initialised model. The full pipeline (data download, training, evaluation) was not run.
>
> **Attach:** Kaggle dataset `oipranesh/cs781-phase1-v3-encoder-modernbert` (`final_model.pt` + `legal_bpe_32k_mask.json`). **Secrets:** `HF_TOKEN` (the course data are gated). **Settings:** Internet ON, GPU T4 x1.


In [ ]:
import builtins
def display(*a, **k):
    for x in a: print(x)
builtins.display = display


In [ ]:
import os
ROOT = os.environ.get('CS781_ROOT', '/kaggle/working'); os.makedirs(ROOT, exist_ok=True)
ARCH_LOADERS_SRC = '"""arch_loaders.py - load any of the four pre-trained architectures as a backbone for classification / tagging.\n   load_backbone(path) -> (backbone, hidden_size, causal, max_len, family)\n   backbone(input_ids, attention_mask) -> object with .last_hidden_state of shape [B, T, hidden_size]"""\nimport torch, torch.nn as nn\nfrom types import SimpleNamespace\nfrom transformers import GPT2Config, GPT2LMHeadModel, ModernBertConfig, ModernBertForMaskedLM, T5Config, T5ForConditionalGeneration\n\nclass BiLSTMLM(nn.Module):\n    def __init__(self, vocab, emb=512, hidden=1024, layers=3):\n        super().__init__()\n        self.vocab, self.emb_dim, self.hidden, self.layers = vocab, emb, hidden, layers\n        self.emb = nn.Embedding(vocab, emb); self.fwd = nn.LSTM(emb, hidden, layers, batch_first=True); self.bwd = nn.LSTM(emb, hidden, layers, batch_first=True)\n        self.norm_f, self.norm_b = nn.LayerNorm(hidden), nn.LayerNorm(hidden)\n        self.proj_f, self.proj_b = nn.Linear(hidden, emb), nn.Linear(hidden, emb); self.out_bias = nn.Parameter(torch.zeros(vocab))\n    def hidden_states(self, input_ids, attention_mask=None):\n        x = input_ids; B, T = x.shape\n        lengths = attention_mask.sum(1) if attention_mask is not None else torch.full((B,), T, device=x.device)\n        rev = (lengths[:, None] - 1 - torch.arange(T, device=x.device)[None, :]).clamp(min=0)\n        hf, _ = self.fwd(self.emb(x)); hb, _ = self.bwd(self.emb(x.gather(1, rev)))\n        hb = hb.gather(1, rev[:, :, None].expand(-1, -1, hb.shape[-1]))\n        return torch.cat([hf, hb], dim=-1)\n\nclass Backbone(nn.Module):\n    """one calling convention for every family (matches what the Phase 3 SeqClassifier expects)"""\n    def __init__(self, family, inner, embed):\n        super().__init__(); self.family, self.inner, self._embed = family, inner, embed\n    def get_input_embeddings(self): return self._embed\n    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, use_cache=False, **kw):\n        f = self.family\n        if f == "gpt2": h = self.inner(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds, use_cache=False).last_hidden_state\n        else:\n            assert inputs_embeds is None, f"prompt tuning (inputs_embeds) is only wired for the gpt2 family, not {f}"\n            if f in ("modernbert", "t5"): h = self.inner(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state\n            elif f == "bilstm": h = self.inner.hidden_states(input_ids, attention_mask)\n            else: raise ValueError(f)\n        return SimpleNamespace(last_hidden_state=h)\n\nCAUSAL = {"gpt2": True, "modernbert": False, "t5": False, "bilstm": False}\n# modules to adapt with LoRA / LoRMA / IA3 (None = full fine-tuning only). ModernBERT\'s q,k,v are ONE fused linear, \'Wqkv\' (rows: q | k | v).\nADAPTER_TARGETS = {"gpt2": "attn.c_attn (fused Conv1D: q|k|v)", "modernbert": "attn.Wqkv (fused Linear: q|k|v)", "t5": "SelfAttention q, k, v, o (Linear)", "bilstm": None}\n\ndef build_full(mc):\n    f, V, d = mc["family"], mc["vocab_size"], mc["hidden_size"]\n    if f == "gpt2":\n        return GPT2LMHeadModel(GPT2Config(vocab_size=V, n_positions=mc["context_length"], n_ctx=mc["context_length"], n_embd=d, n_layer=mc["num_layers"], n_head=mc["num_heads"], n_inner=mc.get("n_inner", 4 * d),\n                                          resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0))\n    if f == "modernbert":\n        return ModernBertForMaskedLM(ModernBertConfig(vocab_size=V, max_position_embeddings=mc["context_length"], hidden_size=d, intermediate_size=mc["intermediate_size"], num_hidden_layers=mc["num_layers"],\n                                                      num_attention_heads=mc["num_heads"], local_attention=mc["local_attention"], global_attn_every_n_layers=mc["global_attn_every_n_layers"], pad_token_id=0,\n                                                      attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, classifier_pooling="mean", sparse_prediction=True, sparse_pred_ignore_index=-100, reference_compile=False))\n    if f == "t5":\n        return T5ForConditionalGeneration(T5Config(vocab_size=V, d_model=d, d_kv=mc["d_kv"], d_ff=mc["d_ff"], num_layers=mc["num_layers"], num_decoder_layers=mc["num_decoder_layers"], num_heads=mc["num_heads"],\n                                                   feed_forward_proj="gated-gelu", dropout_rate=0.0, tie_word_embeddings=True))\n    if f == "bilstm": return BiLSTMLM(V, emb=mc["emb"], hidden=mc["lstm_hidden"], layers=mc["num_layers"])\n    raise ValueError(f)\n\ndef load_backbone(path, device="cpu"):\n    ck = torch.load(path, map_location="cpu", weights_only=False); mc = ck["model_config"]; full = build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True)\n    f = mc["family"]\n    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte\n    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings\n    elif f == "t5": inner, embed = full.encoder, full.shared\n    elif f == "bilstm": inner, embed = full, full.emb\n    else: raise ValueError(f)\n    return Backbone(f, inner, embed).to(device), mc["hidden_size"], CAUSAL[f], mc["context_length"], f\n'
open(ROOT + '/arch_loaders.py', 'w', encoding='utf-8').write(ARCH_LOADERS_SRC)   # loader for the four Phase 1 architectures
CAP_MIN = 60   # minutes of training (a time cap, as in the other notebooks)


In [ ]:

HF_TOKEN = None      # cell 5 falls back to the HF_TOKEN environment variable / Kaggle secret
DURABLE_PUBLISH_ENABLED = False; PUBLISH_EVERY_N_EPOCHS = 999; PUBLISH_AT_EACH_EXPERIMENT_END = False; KAGGLE_DATASET_SLUG = ""     # no mid-run dataset uploads in this notebook


In [ ]:
# ================= CELL 1 — environment, paths, helpers =================
import os, sys, json, math, time, random, re, copy, gc, glob, zipfile, shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from huggingface_hub import snapshot_download, HfApi

SEED = 42

def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device,
      torch.cuda.get_device_name(0) if device.type == "cuda"
      else "(no GPU: training will be slow)")

WORK = Path("/kaggle/working/phase3_rr")
(WORK / "runs").mkdir(parents=True, exist_ok=True)
LEDGER = WORK / "ledger.csv"



# ---------------- PERIODIC KAGGLE DATASET PUBLISHER ----------------
PUBLISH_DIR = Path("/kaggle/working/phase3_publish")
_last_publish_epoch = {}

def _prepare_recovery_upload(reason="periodic"):
    """Stage current recovery checkpoints as ordinary files inside nested folders."""
    if PUBLISH_DIR.exists():
        shutil.rmtree(PUBLISH_DIR)
    PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

    runs_src = WORK / "runs"
    runs_dst = PUBLISH_DIR / "runs"
    runs_dst.mkdir(parents=True, exist_ok=True)
    copied = []
    if runs_src.exists():
        for run_dir in runs_src.iterdir():
            if not run_dir.is_dir():
                continue
            dst = runs_dst / run_dir.name
            dst.mkdir(parents=True, exist_ok=True)
            for filename in ("latest.pt", "best.pt", "result.json"):
                file = run_dir / filename
                if file.is_file():
                    shutil.copy2(file, dst / filename)
                    copied.append(dst / filename)

    for filename in ("ledger.csv", "prediction.json", "prediction_CL.json", "prediction_IT.json"):
        file = WORK / filename
        if file.is_file():
            shutil.copy2(file, PUBLISH_DIR / filename)

    if not any(p.name in ("latest.pt", "best.pt") for p in copied):
        raise RuntimeError(
            "No latest.pt/best.pt checkpoints found to publish. "
            "Check WORK/runs before uploading."
        )

    metadata = {
        "title": "CS781 Phase 3 Recovery Checkpoints",
        "id": KAGGLE_DATASET_SLUG,
        "licenses": [{"name": "CC0-1.0"}],
        "isPrivate": True,
    }
    (PUBLISH_DIR / "dataset-metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    (PUBLISH_DIR / "publish_manifest.json").write_text(json.dumps({
        "project": "CS781 Phase 3 Rhetorical Role Prediction",
        "published_unix": time.time(),
        "reason": reason,
        "dataset_slug": KAGGLE_DATASET_SLUG,
        "checkpoint_files_staged": len(copied),
        "note": "Contains latest/best checkpoints and experiment summaries; full logits/embeddings are not included in periodic versions."
    }, indent=2), encoding="utf-8")

    total_bytes = sum(p.stat().st_size for p in PUBLISH_DIR.rglob("*") if p.is_file())
    print(f"[durable backup] Staged {len(copied)} checkpoint/result files "
          f"({total_bytes / (1024**3):.2f} GiB total).")

def publish_recovery_dataset(reason="periodic"):
    """Publish a new version of the already-created recovery Dataset."""
    if not DURABLE_PUBLISH_ENABLED:
        print("[durable backup] Skipped: Kaggle credentials not configured.")
        return False

    cli = shutil.which("kaggle")
    if cli is None:
        print("[durable backup] Installing Kaggle CLI...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        cli = shutil.which("kaggle")
    if cli is None:
        raise RuntimeError("Kaggle CLI installation did not provide the 'kaggle' command.")

    env = os.environ.copy()
    env["KAGGLE_USERNAME"] = os.environ["KAGGLE_USERNAME"]
    env["KAGGLE_KEY"] = os.environ["KAGGLE_KEY"]

    _prepare_recovery_upload(reason)

    # Dataset is already created. Do NOT use `datasets status` here:
    # status returned 403 in this environment despite list/create working.
    cmd = [
        cli, "datasets", "version",
        "-p", str(PUBLISH_DIR),
        "-m", f"CS781 recovery update: {reason}",
        "--dir-mode", "zip",
    ]
    print("[durable backup] Publishing to", KAGGLE_DATASET_SLUG, "| reason:", reason)
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-5000:])
    if result.stderr:
        print(result.stderr[-2500:])

    combined_output = (result.stdout or "") + "\n" + (result.stderr or "")
    if result.returncode != 0 or any(
        marker in combined_output.lower()
        for marker in ("dataset creation error", "skipping folder:", "403 client error", "forbidden")
    ):
        raise RuntimeError(
            "Kaggle did not confirm a clean dataset version upload. "
            "Inspect the CLI output above and check the dataset page."
        )
    if "dataset version is being created" not in combined_output.lower() and \
       "upload successful" not in combined_output.lower():
        raise RuntimeError(
            "Kaggle CLI exited without a recognizable upload confirmation. "
            "Verify the dataset page before assuming the backup succeeded."
        )

    print("[durable backup] Upload submitted. Confirm the new version and checkpoint files on Kaggle.")
    return True


def find_files(names, roots=("/kaggle/input", "/kaggle/working", "/content", ".")):
    found = {}
    for root in roots:
        if not os.path.exists(root):
            continue
        for n in names:
            if n in found:
                continue
            hits = sorted(glob.glob(f"{root}/**/{n}", recursive=True))
            if hits:
                found[n] = hits[0]
    return found

NEED = [
    "CL_train.json", "CL_dev.json", "CL_test.json",
    "IT_train.json", "IT_dev.json", "IT_test.json",
    "label_vocab.json"
]

paths = find_files(NEED)
missing = [n for n in NEED if n not in paths]

if missing:
    print("Missing locally:", missing)
    print("Trying Hugging Face dataset download...")

    token = os.environ.get("HF_TOKEN") or HF_TOKEN

    # Kaggle Secrets fallback
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None

    if not token:
        raise RuntimeError(
            "HF_TOKEN was not found. In Kaggle, open Add-ons > Secrets, "
            "add a secret named HF_TOKEN with your Hugging Face READ token, "
            "and enable notebook access. Also accept the dataset conditions "
            "on the Hugging Face dataset page. Alternatively, attach the "
            "Phase_3_RR dataset to this Kaggle notebook."
        )

    try:
        local = snapshot_download(
            repo_id="Exploration-Lab/CS-781-Capstone",
            repo_type="dataset",
            token=token,
            allow_patterns=["Phase_3_RR/**"],
            local_dir=str(WORK / "hf_data")
        )
    except Exception as e:
        raise RuntimeError(
            "Hugging Face download failed. Check that you have accepted "
            "the dataset conditions and that HF_TOKEN is a valid token "
            "with read access. Original error: " + repr(e)
        ) from e

    paths = find_files(NEED, roots=(local, str(WORK / "hf_data")))
    missing = [n for n in NEED if n not in paths]

    if missing:
        # Fallback search if the files were placed in nested folders
        paths.update(find_files(missing, roots=(str(WORK / "hf_data"), local)))
        missing = [n for n in NEED if n not in paths]

    if missing:
        raise FileNotFoundError(
            f"Download completed, but these files were not found: {missing}. "
            "Check the dataset's folder structure."
        )

for k, v in paths.items():
    print(f"{k:18s} {v}")

print("Cell 1 setup complete.")

# Load the discovered files once. The parser cell below will convert them into DOCS.
RAW = {name: json.loads(Path(path).read_text(encoding="utf-8")) for name, path in paths.items()}
print("Loaded raw JSON files:", {k: type(v).__name__ for k, v in RAW.items()})


In [ ]:
# ================= CELL 3 — PARSE PHASE 3 DATA (TRAIN / DEV / TEST) =================
# Supports the observed test schema {doc_id: [sentence, ...]} and common labeled schemas.

LABELS_FALLBACK = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute",
    "Dissent", "PrecedentReliedUpon", "PrecedentNotReliedUpon",
    "PrecedentOverruled", "RulingByLowerCourt", "RatioOfTheDecision",
    "RulingByPresentCourt", "None"
]

def _norm_key(k):
    return str(k).strip().lower().replace("-", "_").replace(" ", "_")

def _find_sentence_list(payload):
    if isinstance(payload, list):
        if all(isinstance(x, str) for x in payload):
            return payload
        if payload and all(isinstance(x, dict) for x in payload):
            out = []
            for item in payload:
                sent = None
                for k, v in item.items():
                    if _norm_key(k) in {"text", "sentence", "sent", "content"} and isinstance(v, str):
                        sent = v
                        break
                if sent is None:
                    out = []
                    break
                out.append(sent)
            if out:
                return out
        for item in payload:
            got = _find_sentence_list(item)
            if got is not None:
                return got
        return None

    if isinstance(payload, dict):
        # Prefer conventional keys first.
        for k, v in payload.items():
            if _norm_key(k) in {"text", "texts", "sentences", "sentence", "sents", "content"}:
                if isinstance(v, list) and all(isinstance(x, str) for x in v):
                    return v
                if isinstance(v, str):
                    return [v]
        # Search nested values, but don't treat a labels list as sentence text.
        for k, v in payload.items():
            if _norm_key(k) in {"labels", "label", "y", "tags"}:
                continue
            if isinstance(v, (dict, list)):
                got = _find_sentence_list(v)
                if got is not None:
                    return got
    return None

def _find_label_list(payload):
    if isinstance(payload, dict):
        for k, v in payload.items():
            if _norm_key(k) in {"labels", "label", "tags", "roles", "y"} and isinstance(v, list):
                return v
        for k, v in payload.items():
            if isinstance(v, (dict, list)):
                got = _find_label_list(v)
                if got is not None:
                    return got
    elif isinstance(payload, list):
        if payload and all(isinstance(x, dict) for x in payload):
            vals = []
            for item in payload:
                lab = None
                for k, v in item.items():
                    if _norm_key(k) in {"label", "labels", "tag", "role"} and isinstance(v, str):
                        lab = v
                        break
                if lab is None:
                    vals = []
                    break
                vals.append(lab)
            if vals:
                return vals
        for item in payload:
            if isinstance(item, (dict, list)):
                got = _find_label_list(item)
                if got is not None:
                    return got
    return None

def parse_split(obj, style=None):
    if isinstance(obj, dict):
        items = list(obj.items())
    elif isinstance(obj, list):
        items = [(str(i), x) for i, x in enumerate(obj)]
    else:
        raise ValueError(f"Unsupported split JSON type: {type(obj).__name__}")

    docs = []
    for did, payload in items:
        # The test files were confirmed to map document IDs directly to sentence strings.
        if isinstance(payload, list) and all(isinstance(x, str) for x in payload):
            docs.append({"id": str(did), "sents": payload, "labels": None, "setter": None})
            continue

        sents = _find_sentence_list(payload)
        labels = _find_label_list(payload)
        if sents is None:
            raise ValueError(
                f"{did}: cannot find a sentence list. Payload type={type(payload).__name__}; "
                f"keys={list(payload.keys())[:30] if isinstance(payload, dict) else 'n/a'}"
            )
        if labels is not None:
            labels = [str(x) for x in labels]
            if len(labels) != len(sents):
                raise ValueError(f"{did}: {len(sents)} sentences but {len(labels)} labels")
        docs.append({"id": str(did), "sents": [str(x) for x in sents],
                     "labels": labels, "setter": None})
    return docs, style

# Load label vocabulary if it is a recognizable format; otherwise use the official label names.
_vocab_obj = RAW.get("label_vocab.json")
LABELS = None
if isinstance(_vocab_obj, list) and all(isinstance(x, str) for x in _vocab_obj):
    LABELS = _vocab_obj
elif isinstance(_vocab_obj, dict):
    for key in ("labels", "label_vocab", "id2label", "label2id"):
        value = _vocab_obj.get(key)
        if isinstance(value, list) and all(isinstance(x, str) for x in value):
            LABELS = value
            break
        if isinstance(value, dict) and value:
            try:
                LABELS = [k for k, v in sorted(value.items(), key=lambda kv: int(kv[1]))]
            except Exception:
                LABELS = list(value.keys())
            break
if not LABELS:
    LABELS = LABELS_FALLBACK
LABEL2ID = {lab: i for i, lab in enumerate(LABELS)}
STYLE = {"CL": None, "IT": None}

DOCS = {"CL": {}, "IT": {}}
TEST_OBJ = {}
for dom in ("CL", "IT"):
    for split in ("train", "dev"):
        DOCS[dom][split], _ = parse_split(RAW[f"{dom}_{split}.json"])
        if not DOCS[dom][split]:
            raise ValueError(f"{dom} {split} split is empty")
        for d in DOCS[dom][split]:
            if d["labels"] is None:
                raise ValueError(f"{dom} {split} document {d['id']} has no labels")
            bad = sorted(set(d["labels"]) - set(LABELS))
            if bad:
                raise ValueError(f"{dom} {split} document {d['id']} has labels absent from vocabulary: {bad}")
    TEST_OBJ[dom] = RAW[f"{dom}_test.json"]
    DOCS[dom]["test"], _ = parse_split(TEST_OBJ[dom])
    print(dom, {sp: len(DOCS[dom][sp]) for sp in ("train", "dev", "test")})
    print("  sample sentence counts:", [len(d["sents"]) for d in DOCS[dom]["test"][:2]])

print("Labels:", LABELS)
print("DOCS and LABEL2ID are ready.")


# Validate schema and labels before training.
assert set(("CL", "IT")).issubset(DOCS)
for _dom in ("CL", "IT"):
    for _split in ("train", "dev"):
        assert DOCS[_dom][_split], f"Empty {_dom}/{_split}"
        for _d in DOCS[_dom][_split]:
            assert _d["labels"] is not None, f"Missing labels in {_dom}/{_split}/{_d['id']}"
            assert len(_d["sents"]) == len(_d["labels"]), f"Sentence/label mismatch in {_dom}/{_split}/{_d['id']}"
    for _d in DOCS[_dom]["test"]:
        assert _d["labels"] is None, f"Expected unlabeled test doc: {_d['id']}"
print("Train/dev/test schema validation passed.")


In [ ]:
# ================= CELL 5 — tokenizer, example builder (sentence-only vs context-aware), batching =================
from tokenizers import Tokenizer
_tok_files = find_files(["legal_bpe_32k_mask.json"])
if "legal_bpe_32k_mask.json" not in _tok_files:
    raise FileNotFoundError(
        "legal_bpe_32k_mask.json was not found. Attach the Phase 1 tokenizer dataset "
        "to Kaggle or update tok_path to its actual path."
    )
tok_path = _tok_files["legal_bpe_32k_mask.json"]
tokenizer = Tokenizer.from_file(tok_path)
print("tokenizer:", tok_path, "| vocab:", tokenizer.get_vocab_size())

_ENC = {}
def enc(text):
    r = _ENC.get(text)
    if r is None: r = _ENC[text] = tokenizer.encode(text, add_special_tokens=False).ids
    return r

MK = {"prev": enc(" <<previous>> "), "next": enc(" <<next>> "), "tgt": enc(" <<target>> ")}
SEP = enc(" | ")

def build_examples(docs, ctx, k_left=3, k_right=1, t_budget=96, l_budget=160, r_budget=64, max_sent=96):
    """One example per sentence. Context-aware input (for a causal LM the target comes LAST so it can attend to everything):
       <<previous>> left sentences <<next>> right sentences <<target>> target sentence."""
    exs = []
    for d in docs:
        toks = [enc(s)[:max_sent] for s in d["sents"]]
        n = len(toks)
        for i in range(n):
            tgt = toks[i][:t_budget]
            if ctx:
                left, right = [], []
                for j in range(max(0, i - k_left), i): left += toks[j] + SEP
                for j in range(i + 1, min(n, i + 1 + k_right)): right += toks[j] + SEP
                ids = MK["prev"] + left[-l_budget:] + MK["next"] + right[:r_budget] + MK["tgt"] + tgt
            else:
                ids = list(tgt)
            if not ids: ids = [0]
            rel = i / max(n - 1, 1)
            y = LABEL2ID[d["labels"][i]] if d["labels"] is not None and d["labels"][i] is not None else -1
            exs.append(dict(ids=ids, pos=[rel, rel * rel], y=y, doc=d["id"], i=i))
    return exs

def make_batches(exs, bs, shuffle, rng):
    idx = list(range(len(exs)))
    if shuffle: rng.shuffle(idx)
    batches = []
    for c in range(0, len(idx), bs * 50):
        ch = sorted(idx[c:c + bs * 50], key=lambda j: len(exs[j]["ids"]))
        batches += [ch[k:k + bs] for k in range(0, len(ch), bs)]
    if shuffle: rng.shuffle(batches)
    return batches

def collate(exs, bidx):
    import torch
    L = max(len(exs[j]["ids"]) for j in bidx)
    ids = torch.zeros((len(bidx), L), dtype=torch.long); mask = torch.zeros_like(ids)
    for r, j in enumerate(bidx):
        n = len(exs[j]["ids"]); ids[r, :n] = torch.tensor(exs[j]["ids"]); mask[r, :n] = 1
    pos = torch.tensor([exs[j]["pos"] for j in bidx], dtype=torch.float32)
    y = torch.tensor([exs[j]["y"] for j in bidx], dtype=torch.long)
    return ids, mask, pos, y


In [ ]:
# ================= CELL 4 — parser status =================
# Do not redefine parse_split here. Cell 3 is the single source of truth.
print("DOCS keys:", list(DOCS.keys()))
for _dom in ("CL", "IT"):
    print(_dom, {k: len(v) for k, v in DOCS[_dom].items()})


In [ ]:
# ================= CELL 7a — metrics + inference-time post-processing (numpy only) =================
def prf_report(y_true, y_pred, K):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    cm = np.zeros((K, K), dtype=np.int64); np.add.at(cm, (y_true, y_pred), 1)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros(K), where=(tp + fp) > 0)
    rec  = np.divide(tp, tp + fn, out=np.zeros(K), where=(tp + fn) > 0)
    f1   = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
    present = (cm.sum(0) + cm.sum(1)) > 0          # labels in gold or prediction = sklearn's default macro average
    return dict(macro_f1=float(f1[present].mean()), macro_p=float(prec[present].mean()), macro_r=float(rec[present].mean()),
                macro_f1_all=float(f1.mean()), acc=float(tp.sum() / cm.sum()), f1=f1, prec=prec, rec=rec,
                support=cm.sum(1), cm=cm, n_present=int(present.sum()))

def softmax_np(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def transition_logprobs(docs, label2id, K, alpha=1.0):
    T = np.full((K, K), alpha); S = np.full(K, alpha)
    for d in docs:
        ys = [label2id[l] for l in d["labels"]]
        S[ys[0]] += 1
        for a, b in zip(ys[:-1], ys[1:]): T[a, b] += 1
    return np.log(T / T.sum(1, keepdims=True)), np.log(S / S.sum())

def viterbi_smooth(logp, logT, logS, lam):
    """argmax_y  sum_i logp[i,y_i] + lam * (logS[y_0] + sum_i logT[y_{i-1}, y_i])"""
    n, K = logp.shape
    if lam == 0 or n == 1: return logp.argmax(1)
    dp = np.zeros((n, K)); bp = np.zeros((n, K), dtype=int)
    dp[0] = logp[0] + lam * logS
    for i in range(1, n):
        cand = dp[i - 1][:, None] + lam * logT
        bp[i] = cand.argmax(0); dp[i] = cand.max(0) + logp[i]
    path = [int(dp[-1].argmax())]
    for i in range(n - 1, 0, -1): path.append(int(bp[i, path[-1]]))
    return np.array(path[::-1])

def group_by_doc(exs):
    g = {}
    for j, e in enumerate(exs): g.setdefault(e["doc"], []).append(j)
    return {d: sorted(ix, key=lambda j: exs[j]["i"]) for d, ix in g.items()}

def smooth_split(probs, exs, logT, logS, lam):
    pred = np.zeros(len(exs), dtype=int)
    for d, ix in group_by_doc(exs).items():
        pred[ix] = viterbi_smooth(np.log(np.clip(probs[ix], 1e-9, 1)), logT, logS, lam)
    return pred

def knn_interpolate(q_emb, tr_emb, tr_y, q_probs, K, k=10, tau=0.1, lam=0.5):
    """p = lam * p_model + (1-lam) * p_kNN  (neighbourhood-at-inference idea; neighbours come from TRAIN sentences only)"""
    qe = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-9)
    te = tr_emb / (np.linalg.norm(tr_emb, axis=1, keepdims=True) + 1e-9)
    sims = qe @ te.T
    top = np.argpartition(-sims, kth=min(k, sims.shape[1] - 1), axis=1)[:, :k]
    out = np.zeros_like(q_probs)
    for r in range(len(qe)):
        s = sims[r, top[r]] / tau; w = softmax_np(s)
        np.add.at(out[r], tr_y[top[r]], w)
    return lam * q_probs + (1 - lam) * out

def validate_submission(obj, docs, label_set, style=None):
    """every document and sentence present, order preserved, exactly one valid label per sentence"""
    d2 = {d["id"]: d for d in docs}
    problems = []
    parsed, _ = parse_split(obj, style)
    if [d["id"] for d in parsed] != [d["id"] for d in docs]: problems.append("document ids / order differ from the test file")
    for p in parsed:
        ref = d2.get(p["id"])
        if ref is None: continue
        if p["sents"] != ref["sents"]: problems.append(f"{p['id']}: sentence text/order changed")
        if p["labels"] is None or len(p["labels"]) != len(ref["sents"]): problems.append(f"{p['id']}: label count mismatch"); continue
        bad = [l for l in p["labels"] if l not in label_set]
        if bad: problems.append(f"{p['id']}: invalid labels {set(bad)}")
    return problems


In [ ]:
# ================= CELL 6 — backbones, classifier, adapters (LoRA / rsLoRA / LoRMA+ / LoRMAπ / IA3 / QLoRA / prompt) =================
from transformers import GPT2Config, GPT2Model
from transformers.pytorch_utils import Conv1D

# ---- architecture registry: exact known 3,000-step Phase 1 checkpoint ----
PHASE1_CHECKPOINT = "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"
ARCHS = {}   # the earlier GPT-2 checkpoint is not needed here; the Phase 1 model of the chosen family is registered in the next cell
print("Registered architecture:", ARCHS)

def build_backbone(arch, bb_dropout=0.1, n_layers=None):
    """-> (backbone, hidden, causal, n_positions, family). n_layers: keep only the first n layers (student initialisation)."""
    spec = ARCHS[arch]
    if spec["family"] != "gpt2":
        raise NotImplementedError(f"No loader for family '{spec['family']}'. Run the inventory cell, paste its output, and add a loader here.")
    ck = torch.load(spec["path"], map_location="cpu", weights_only=False)
    cfg = ck.get("model_config")
    if not isinstance(cfg, dict):
        raise ValueError(f"Checkpoint has no model_config dictionary: {spec['path']}")
    vocab_size = cfg.get("vocab_size")
    context_length = cfg.get("context_length", cfg.get("n_positions"))
    hidden_size = cfg.get("hidden_size", cfg.get("n_embd"))
    num_layers = cfg.get("num_layers", cfg.get("n_layer"))
    num_heads = cfg.get("num_heads", cfg.get("n_head"))
    if not all(v is not None for v in (vocab_size, context_length, hidden_size, num_layers, num_heads)):
        raise ValueError(f"Incomplete model configuration: {cfg}")
    gcfg = GPT2Config(
        vocab_size=vocab_size, n_positions=context_length, n_ctx=context_length,
        n_embd=hidden_size, n_layer=num_layers, n_head=num_heads,
        resid_pdrop=bb_dropout, embd_pdrop=bb_dropout, attn_pdrop=bb_dropout
    )
    bb = GPT2Model(gcfg)
    sd = ck.get("model_state_dict", ck)
    if not isinstance(sd, dict):
        raise ValueError("model_state_dict is not a dictionary")
    prefixed = {k[len("transformer."):]: v for k, v in sd.items() if k.startswith("transformer.")}
    if prefixed:
        sd = prefixed
    sd = {k: v for k, v in sd.items() if not k.startswith("lm_head.")}
    missing, unexpected = bb.load_state_dict(sd, strict=False)
    if missing or unexpected:
        raise RuntimeError(
            f"Checkpoint/backbone mismatch. Missing keys (first 8): {missing[:8]}; "
            f"unexpected keys (first 8): {unexpected[:8]}"
        )
    del ck
    if n_layers is not None and n_layers < len(bb.h):
        bb.h = nn.ModuleList(list(bb.h)[:n_layers]); bb.config.n_layer = n_layers
    return bb, cfg["hidden_size"], True, cfg["context_length"], "gpt2"

class SeqClassifier(nn.Module):
    def __init__(self, backbone, hidden, n_labels, causal=True, pos_dim=2, dropout=0.1, family="gpt2"):
        super().__init__()
        self.backbone, self.causal, self.pos_dim, self.family = backbone, causal, pos_dim, family
        self.drop = nn.Dropout(dropout); self.head = nn.Linear(hidden + pos_dim, n_labels); self.prompt = None
    def forward(self, ids, mask, pos=None, return_emb=False):
        B = ids.size(0); lengths = mask.sum(1)
        if self.prompt is not None:                       # prompt tuning: learnable vectors in front of the input
            emb = self.backbone.get_input_embeddings()(ids)
            P = self.prompt.shape[0]
            emb = torch.cat([self.prompt.unsqueeze(0).expand(B, -1, -1).to(emb.dtype), emb], 1)
            mask = torch.cat([torch.ones(B, P, device=ids.device, dtype=mask.dtype), mask], 1); lengths = lengths + P
            h = self.backbone(inputs_embeds=emb, attention_mask=mask, use_cache=False).last_hidden_state
        else:
            h = self.backbone(input_ids=ids, attention_mask=mask, use_cache=False).last_hidden_state
        if self.causal: pooled = h[torch.arange(B, device=h.device), lengths - 1]       # last real token sees the whole input
        else:
            m = mask.unsqueeze(-1).to(h.dtype); pooled = (h * m).sum(1) / m.sum(1)
        feats = pooled if (self.pos_dim == 0 or pos is None) else torch.cat([pooled, pos.to(pooled.dtype)], -1)
        logits = self.head(self.drop(feats))
        return (logits, pooled) if return_emb else logits

def count_params(m):
    tot = sum(p.numel() for p in m.parameters()); tr = sum(p.numel() for p in m.parameters() if p.requires_grad); return tot, tr
def model_size_mb(m):
    return (sum(p.numel() * p.element_size() for p in m.parameters()) + sum(b.numel() * b.element_size() for b in m.buffers())) / 2**20

# ---------------- adapter building blocks ----------------
class LoRADelta(nn.Module):                # additive:  dW x = (alpha/r) B A x   (rsLoRA: alpha/sqrt(r))
    def __init__(self, nx, nf, r, alpha, drop, rs=False):
        super().__init__()
        self.A = nn.Parameter(torch.empty(nx, r).uniform_(-1 / math.sqrt(nx), 1 / math.sqrt(nx))); self.B = nn.Parameter(torch.zeros(r, nf))
        self.scale = alpha / (math.sqrt(r) if rs else r); self.drop = nn.Dropout(drop)
    def forward(self, x): return (self.drop(x) @ self.A) @ self.B * self.scale

class LoRMAPlus(nn.Module):                # multiplicative, additive rank inflation: h' = (I + (alpha/r) B A) h ; B=0 at init
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d))); self.B = nn.Parameter(torch.zeros(d, r)); self.scale = alpha / r
    def forward(self, h): return h + ((h @ self.A.t()) @ self.B.t()) * self.scale      # = W0x + (alpha/r) B A (W0 x)

class LoRMAPi(nn.Module):                  # multiplicative, permutation-based rank inflation: h' = I_pi(BA) h, I_pi(BA) = I at init
    def __init__(self, d, r):
        super().__init__()
        B = torch.empty(d, r).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d)); B[:, 0] = 1.0
        A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A)
        i = torch.arange(d); self.register_buffer("idx", (i[None, :] - i[:, None]) % d)   # row i rotated right by i
    def matrix(self): return (self.B @ self.A).gather(1, self.idx)
    def forward(self, h): return h @ self.matrix().t().to(h.dtype)

class QVAdapt(nn.Module):                  # wraps GPT-2's fused c_attn (q|k|v); adapts q and v (k and v for IA3)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__()
        nx, nf = conv_dims(base); d = nf // 3; self.base, self.kind, self.d = base, kind, d
        if kind == "lora":         self.q, self.v = LoRADelta(nx, d, r, alpha, drop, rs), LoRADelta(nx, d, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.q, self.v = LoRMAPlus(d, r, alpha), LoRMAPlus(d, r, alpha)
        elif kind == "lorma_pi":   self.q, self.v = LoRMAPi(d, r), LoRMAPi(d, r)
        elif kind == "ia3":        self.lk, self.lv = nn.Parameter(torch.ones(d)), nn.Parameter(torch.ones(d))
        else: raise ValueError(kind)
    def forward(self, x):
        out = self.base(x); d = self.d; q, k, v = out[..., :d], out[..., d:2 * d], out[..., 2 * d:]
        if self.kind == "lora":            q, v = q + self.q(x), v + self.v(x)
        elif self.kind.startswith("lorma"): q, v = self.q(q), self.v(v)
        else:                              k, v = k * self.lk, v * self.lv
        return torch.cat([q, k, v], -1)

class IA3Input(nn.Module):                 # IA3 on the FFN: rescale the activation entering the down-projection
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(conv_dims(base)[0]))
    def forward(self, x): return self.base(x * self.l)

class LinAdapt(nn.Module):                 # same adapters for plain nn.Linear projections (non-GPT-2 backbones)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); self.base, self.kind = base, kind; di, do = base.in_features, base.out_features
        if kind == "lora": self.op = LoRADelta(di, do, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.op = LoRMAPlus(do, r, alpha)
        elif kind == "lorma_pi": self.op = LoRMAPi(do, r)
        elif kind == "ia3": self.l = nn.Parameter(torch.ones(do))
    def forward(self, x):
        y = self.base(x)
        if self.kind == "lora": return y + self.op(x)
        if self.kind == "ia3": return y * self.l
        return self.op(y)

def wrap_linear_targets(root, kind, names, **kw):
    n = 0
    for _, parent in list(root.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(parent, cname, LinAdapt(child, kind, **kw)); n += 1
    assert n > 0, f"no nn.Linear named {names} found in the backbone"

# ---------------- NF4 quantisation (QLoRA-style frozen base weights, 4-bit packed) ----------------
NF4_CODE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
    -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])
def nf4_quantize(w, block=64):
    flat = w.detach().float().reshape(-1); n = flat.numel(); pad = (-n) % block
    if pad: flat = F.pad(flat, (0, pad))
    blocks = flat.view(-1, block); absmax = blocks.abs().amax(dim=1).clamp_min(1e-8)
    normed = (blocks / absmax[:, None]).reshape(-1)
    code = NF4_CODE.to(flat.device); mids = (code[:-1] + code[1:]) / 2
    idx = torch.bucketize(normed, mids).to(torch.uint8)
    return ((idx[0::2] << 4) | idx[1::2]), absmax.half(), n
def nf4_dequant(packed, absmax, n, shape, block=64):
    code = NF4_CODE.to(packed.device)
    idx = torch.stack([(packed >> 4).long(), (packed & 0x0F).long()], dim=1).reshape(-1)
    return (code[idx].view(-1, block) * absmax.float()[:, None]).reshape(-1)[:n].view(shape)

class NF4Conv1D(nn.Module):
    def __init__(self, conv, block=64):
        super().__init__(); nx, nf = conv.weight.shape; self.nx, self.nf, self.block = nx, nf, block
        packed, absmax, n = nf4_quantize(conv.weight.data, block); self.n = n
        self.register_buffer("packed", packed); self.register_buffer("absmax", absmax); self.register_buffer("bias", conv.bias.data.clone())
    def forward(self, x):
        W = nf4_dequant(self.packed, self.absmax, self.n, (self.nx, self.nf), self.block).to(x.dtype)
        return x @ W + self.bias.to(x.dtype)

def conv_dims(m):
    if isinstance(m, NF4Conv1D): return m.nx, m.nf
    return tuple(m.weight.shape)                     # HF Conv1D weight is [in, out]

METHODS = ("full", "lora", "rslora", "qlora", "lorma_plus", "lorma_pi", "ia3", "prompt")
def apply_method(model, method, hp):
    assert method in METHODS, method
    if method == "full":
        for p in model.parameters(): p.requires_grad = True
        return model
    for p in model.backbone.parameters(): p.requires_grad = False
    r, alpha, drop = hp.get("r", 8), hp.get("alpha", 16), hp.get("adapter_dropout", 0.05)
    if method == "prompt":
        emb = model.backbone.get_input_embeddings().weight; P = hp.get("n_prompt", 20)
        init = emb[torch.randint(0, emb.shape[0], (P,))].detach().clone()
        model.prompt = nn.Parameter(init); return model
    kind = {"lora": "lora", "rslora": "lora", "qlora": "lora", "lorma_plus": "lorma_plus", "lorma_pi": "lorma_pi", "ia3": "ia3"}[method]
    kw = dict(r=r, alpha=alpha, drop=drop, rs=(method == "rslora"))
    if model.family == "gpt2":
        for blk in model.backbone.h:
            if method == "qlora":
                for owner, name in ((blk.attn, "c_attn"), (blk.attn, "c_proj"), (blk.mlp, "c_fc"), (blk.mlp, "c_proj")):
                    setattr(owner, name, NF4Conv1D(getattr(owner, name)))
            assert isinstance(blk.attn.c_attn, (Conv1D, NF4Conv1D)), "unexpected c_attn type in this transformers version"
            blk.attn.c_attn = QVAdapt(blk.attn.c_attn, kind, **kw)
            if kind == "ia3": blk.mlp.c_proj = IA3Input(blk.mlp.c_proj)
    else:
        if method == "qlora": raise NotImplementedError("QLoRA is implemented for the GPT-2 family only")
        names = ("k_proj", "v_proj") if kind == "ia3" else ("q_proj", "v_proj")
        wrap_linear_targets(model.backbone, kind, names, **({} if kind == "ia3" else kw))
    return model


In [ ]:
# ================= CELL 8 — training / evaluation harness + RESUMABLE CHECKPOINTS =================
# Recovery strategy:
#   latest.pt: full model + optimizer/scheduler/scaler + epoch + RNG state (resume after interruption)
#   best.pt: best validation model weights (use for inference)
#   result.json/logits/embeddings: completed experiment artifacts
K = len(LABELS)
HP_DEFAULT = {
    "full": dict(lr=3e-5), "lora": dict(lr=5e-4, r=8, alpha=16), "rslora": dict(lr=5e-4, r=8, alpha=16), "qlora": dict(lr=5e-4, r=8, alpha=16),
    "lorma_plus": dict(lr=5e-4, r=8, alpha=16), "lorma_pi": dict(lr=5e-4, r=8), "ia3": dict(lr=3e-3), "prompt": dict(lr=1e-2, n_prompt=20),
}
CFG_DEFAULT = dict(arch=None, method="full", ctx=True, epochs=8, patience=3, bs=16, warmup=0.1, wd=0.01, head_lr=1e-3, label_smooth=0.05,
                   class_weights=True, bb_dropout=0.1, k_left=3, k_right=1, seed=SEED, n_layers=None, kd=None, merge_dev=False,
                   fixed_epochs=None, hp={}, force=False, resume=True)
use_amp = device.type == "cuda"

def atomic_torch_save(obj, path):
    """Write checkpoint to a temporary file then atomically replace destination."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def cpu_state_dict(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def checkpoint_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])

@torch.no_grad()
def predict(model, exs, bs=64, emb=False):
    model.eval(); lg = np.zeros((len(exs), K), np.float32); em = None
    order = sorted(range(len(exs)), key=lambda j: len(exs[j]["ids"]))
    for s in range(0, len(order), bs):
        bidx = order[s:s + bs]; ids, mask, pos, _ = collate(exs, bidx)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            out = model(ids.to(device), mask.to(device), pos.to(device), return_emb=emb)
        if emb:
            out, pooled = out
            if em is None: em = np.zeros((len(exs), pooled.shape[-1]), np.float16)
            em[bidx] = pooled.float().cpu().numpy().astype(np.float16)
        lg[bidx] = out.float().cpu().numpy()
    return (lg, em) if emb else lg

def snapshot(model):
    return {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}

def restore(model, snap):
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in snap: p.copy_(snap[n].to(p.device))

def run_one(name, dom, cfg):
    c = {**CFG_DEFAULT, **cfg}
    if not ARCHS:
        raise RuntimeError("ARCHS is empty; configure the Phase 1 checkpoint in Cell 6.")
    c["arch"] = c["arch"] or next(iter(ARCHS))
    hp = {**HP_DEFAULT[c["method"]], **c["hp"]}
    rdir = WORK / "runs" / f"{name}_{dom}"
    rdir.mkdir(parents=True, exist_ok=True)
    latest_path, best_path = rdir / "latest.pt", rdir / "best.pt"
    if (rdir / "result.json").exists() and not c["force"]:
        print(f"[{name}/{dom}] already done - skipping (force=True to redo)")
        return json.load(open(rdir / "result.json", encoding="utf-8"))

    seed_all(c["seed"]); t_all = time.time()
    tr_docs = DOCS[dom]["train"] + (DOCS[dom]["dev"] if c["merge_dev"] else [])
    kw = dict(ctx=c["ctx"], k_left=c["k_left"], k_right=c["k_right"])
    ex_tr = build_examples(tr_docs, **kw)
    ex_dv = build_examples(DOCS[dom]["dev"], **kw)
    ex_te = build_examples(DOCS[dom]["test"], **kw)
    y_tr = np.array([e["y"] for e in ex_tr]); y_dv = np.array([e["y"] for e in ex_dv])

    bb, hidden, causal, npos, fam = build_backbone(c["arch"], c["bb_dropout"], c["n_layers"])
    model = SeqClassifier(bb, hidden, K, causal, 2, 0.1, fam)
    apply_method(model, c["method"], hp); model.to(device)
    longest = max(len(e["ids"]) for e in ex_tr + ex_dv + ex_te) + (hp.get("n_prompt", 0) if c["method"] == "prompt" else 0)
    assert longest <= npos, f"input length {longest} exceeds the model context {npos}"

    tot_p, tr_p = count_params(model); size_mb = model_size_mb(model)
    cnt = np.bincount(y_tr, minlength=K).astype(float); cw = None
    if c["class_weights"]:
        w = 1 / np.sqrt(np.maximum(cnt, 1)); w = w * cnt.sum() / (w * cnt).sum()
        cw = torch.tensor(w, dtype=torch.float32, device=device)
    T_logits = None
    if c["kd"]:
        assert not c["merge_dev"], "KD + merge_dev not supported"
        T_logits = np.load(WORK / "runs" / f"teacher_{dom}.npy")
        assert len(T_logits) == len(ex_tr), "teacher logits do not match the training examples"

    other = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head.")]
    opt = torch.optim.AdamW(
        [{"params": other, "lr": hp["lr"]},
         {"params": list(model.head.parameters()), "lr": c["head_lr"]}],
        weight_decay=c["wd"])
    epochs = c["fixed_epochs"] or c["epochs"]
    spe = math.ceil(len(ex_tr) / c["bs"]); total = spe * epochs
    warm = int(c["warmup"] * total)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda step: min((step + 1) / max(1, warm),
                              max(0.0, (total - step) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    rng = random.Random(c["seed"])
    trainable = [p for p in model.parameters() if p.requires_grad]

    start_epoch, best, best_ep, bad, hist = 1, -1.0, 0, 0, []
    best_state = None
    if c["resume"] and latest_path.exists() and not c["force"]:
        try:
            ck = torch.load(latest_path, map_location="cpu", weights_only=False)
            model.load_state_dict(ck["model_state"])
            opt.load_state_dict(ck["optimizer_state"])
            sched.load_state_dict(ck["scheduler_state"])
            scaler.load_state_dict(ck["scaler_state"])
            start_epoch = int(ck["epoch"]) + 1
            best, best_ep, bad = float(ck["best_f1"]), int(ck["best_epoch"]), int(ck["bad_epochs"])
            hist = ck.get("history", [])
            best_state = ck.get("best_model_state")
            restore_rng_state(ck.get("rng_state"))
            print(f"[{name}/{dom}] RESUMING from epoch {start_epoch} (latest checkpoint epoch {ck['epoch']})")
        except Exception as e:
            print(f"[{name}/{dom}] WARNING: could not resume checkpoint ({e}); starting fresh.")
            start_epoch, best, best_ep, bad, hist, best_state = 1, -1.0, 0, 0, [], None
    if c["force"]:
        print(f"[{name}/{dom}] force=True: starting a fresh run; old checkpoints will be replaced.")

    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    print(f"[{name}/{dom}] arch={c['arch']} method={c['method']} ctx={c['ctx']} | train {len(ex_tr)} sents | "
          f"params total {tot_p/1e6:.1f}M trainable {tr_p/1e6:.3f}M ({100*tr_p/tot_p:.2f}%) | size {size_mb:.0f} MB")
    t0 = time.time()
    for ep in range(start_epoch, epochs + 1):
        model.train(); tl, nb = 0.0, 0
        for bidx in make_batches(ex_tr, c["bs"], True, rng):
            ids, mask, pos, y = collate(ex_tr, bidx)
            ids, mask, pos, y = ids.to(device), mask.to(device), pos.to(device), y.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(ids, mask, pos)
            loss = F.cross_entropy(logits.float(), y, weight=cw, label_smoothing=c["label_smooth"])
            if T_logits is not None:
                temp, alpha = c["kd"].get("T", 2.0), c["kd"].get("alpha", 0.5)
                tl_ = torch.tensor(T_logits[bidx], device=device)
                kl = F.kl_div(F.log_softmax(logits.float() / temp, -1),
                              F.softmax(tl_ / temp, -1), reduction="batchmean") * temp * temp
                loss = alpha * loss + (1 - alpha) * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item(); nb += 1

        if c["merge_dev"]:
            epoch_f1 = None
            print(f"   epoch {ep}/{epochs} loss {tl/max(nb,1):.4f} (train+dev refit: no validation)")
            best_state = cpu_state_dict(model); best_ep = ep
        else:
            rep_ep = prf_report(y_dv, predict(model, ex_dv).argmax(1), K)
            epoch_f1 = float(rep_ep["macro_f1"])
            hist.append(dict(epoch=ep, loss=tl / max(nb, 1), dev_macro_f1=epoch_f1))
            print(f"   epoch {ep}/{epochs} loss {tl/max(nb,1):.4f} | dev macro-F1 {epoch_f1:.4f} "
                  f"(P {rep_ep['macro_p']:.3f} R {rep_ep['macro_r']:.3f}) acc {rep_ep['acc']:.3f}")
            if epoch_f1 > best + 1e-4:
                best, best_ep, bad = epoch_f1, ep, 0
                best_state = cpu_state_dict(model)
                atomic_torch_save({
                    "model_state": best_state, "epoch": ep, "best_f1": best,
                    "best_epoch": best_ep, "arch": c["arch"], "method": c["method"],
                    "domain": dom, "experiment": name, "config": c,
                }, best_path)
            else:
                bad += 1

        # Save after EVERY completed epoch. A runtime interruption loses at most the in-progress epoch.
        atomic_torch_save({
            "format_version": 1, "experiment": name, "domain": dom, "epoch": ep,
            "model_state": cpu_state_dict(model), "optimizer_state": opt.state_dict(),
            "scheduler_state": sched.state_dict(), "scaler_state": scaler.state_dict(),
            "best_f1": best, "best_epoch": best_ep, "bad_epochs": bad,
            "best_model_state": best_state, "history": hist, "rng_state": checkpoint_rng_state(),
            "config": c, "hp": hp, "saved_unix": time.time(),
        }, latest_path)
        print(f"   checkpoint saved: {latest_path.name} (epoch {ep})")

        # Publish a durable Dataset version every N completed epochs.
        # This runs after local checkpoint files have been safely written.
        if (DURABLE_PUBLISH_ENABLED
                and PUBLISH_EVERY_N_EPOCHS > 0
                and ep % PUBLISH_EVERY_N_EPOCHS == 0
                and _last_publish_epoch.get(str(rdir), 0) != ep):
            publish_recovery_dataset(reason=f"{name}_{dom}_epoch_{ep}")
            _last_publish_epoch[str(rdir)] = ep


        if not c["merge_dev"] and bad >= c["patience"] and not c["fixed_epochs"]:
            print("   early stop"); break

    # If all epochs were already completed in a valid checkpoint, use its best state.
    if best_state is not None:
        model.load_state_dict(best_state)
        atomic_torch_save({
            "model_state": cpu_state_dict(model), "epoch": best_ep or epochs,
            "best_f1": best, "best_epoch": best_ep, "arch": c["arch"],
            "method": c["method"], "domain": dom, "experiment": name,
            "config": c, "saved_unix": time.time(),
        }, best_path)

    train_sec = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else 0.0
    t1 = time.time(); lg_dv, em_dv = predict(model, ex_dv, emb=True)
    sps = len(ex_dv) / max(time.time() - t1, 1e-6)
    lg_te, em_te = predict(model, ex_te, emb=True)
    lg_tr, em_tr = predict(model, ex_tr, emb=True)
    for nm, arr in dict(dev_logits=lg_dv, test_logits=lg_te, train_logits=lg_tr,
                        dev_emb=em_dv, test_emb=em_te, train_emb=em_tr).items():
        np.save(rdir / f"{nm}.npy", arr)
    np.save(rdir / "train_y.npy", y_tr)
    rep = prf_report(y_dv, lg_dv.argmax(1), K)
    res = dict(name=name, domain=dom, arch=c["arch"], method=c["method"], ctx=c["ctx"],
               kd=bool(c["kd"]), merge_dev=c["merge_dev"], n_layers=c["n_layers"],
               best_epoch=best_ep or epochs, epochs_run=len(hist) or epochs,
               dev_macro_f1=rep["macro_f1"], dev_macro_p=rep["macro_p"], dev_macro_r=rep["macro_r"],
               dev_macro_f1_all13=rep["macro_f1_all"], dev_acc=rep["acc"],
               params_total=tot_p, params_trainable=tr_p, trainable_pct=100 * tr_p / tot_p,
               model_size_mb=size_mb, peak_gpu_mb=peak_mb, train_sec=train_sec,
               sents_per_sec=sps, cfg=json.dumps({**c, "hp": hp}, default=str),
               **{f"f1_{l}": float(rep["f1"][i]) for i, l in enumerate(LABELS)})
    tmp_result = rdir / "result.json.tmp"
    with open(tmp_result, "w", encoding="utf-8") as f:
        json.dump({**res, "confusion": rep["cm"].tolist(), "history": hist}, f, indent=2)
    os.replace(tmp_result, rdir / "result.json")
    newrow = pd.DataFrame([res])
    if LEDGER.exists():
        old = pd.read_csv(LEDGER)
        old = old[~((old["name"] == name) & (old["domain"] == dom))]
        newrow = pd.concat([old, newrow], ignore_index=True)
    newrow.to_csv(LEDGER, index=False)
    if PUBLISH_AT_EACH_EXPERIMENT_END and DURABLE_PUBLISH_ENABLED:
        publish_recovery_dataset(reason=f"{name}_{dom}_experiment_complete")
    print(f"[{name}/{dom}] DONE dev macro-F1 {rep['macro_f1']:.4f} @epoch {res['best_epoch']} | "
          f"{train_sec/60:.1f} min | peak {peak_mb:.0f} MB | {sps:.0f} sent/s")
    del model, opt, scaler, best_state
    gc.collect(); torch.cuda.empty_cache()
    return res

def run_experiment(name, domains=("CL", "IT"), **cfg):
    return {d: run_one(name, d, cfg) for d in domains}

def make_teacher(run_names, dom):
    """Geometric-mean ensemble of finished runs' train logits -> teacher for KD."""
    lp = [np.log(softmax_np(np.load(WORK / "runs" / f"{n}_{dom}" / "train_logits.npy")) + 1e-9)
          for n in run_names]
    np.save(WORK / "runs" / f"teacher_{dom}.npy", np.mean(lp, 0).astype(np.float32))
    print(f"teacher_{dom}: ensemble of {run_names}")

def show_ledger(cols=None):
    if not LEDGER.exists():
        print("No completed experiments in ledger yet.")
        return pd.DataFrame()
    df = pd.read_csv(LEDGER).drop_duplicates(subset=["name", "domain"], keep="last")
    cols = cols or ["name", "domain", "arch", "method", "ctx", "kd", "dev_macro_f1",
                    "dev_macro_p", "dev_macro_r", "best_epoch", "params_trainable",
                    "model_size_mb", "peak_gpu_mb", "train_sec"]
    print(df[cols].round(4).to_string(index=False))
    return df


In [ ]:

# ---- registration of the ModernBERT-style encoder: the original pipeline only knew the GPT-2 family; arch_loaders.py (Phase 1 loader) adds the other backbones
FAMILY = "modernbert"
sys.path.insert(0, ROOT); import arch_loaders as AL
CKPT_PATH = None
for p_ in sorted(glob.glob("/kaggle/input/**/final_model.pt", recursive=True)):      # the checkpoint whose model_config family is FAMILY
    try: fam_ = torch.load(p_, map_location="cpu", weights_only=False)["model_config"].get("family", "gpt2")
    except Exception: continue
    if fam_ == FAMILY: CKPT_PATH = p_; break
assert CKPT_PATH, "attach the dataset oipranesh/cs781-phase1-v3-encoder-modernbert (final_model.pt + legal_bpe_32k_mask.json)"
ARCHS[FAMILY] = dict(path=CKPT_PATH, family=FAMILY)
_orig_build_backbone = build_backbone
def build_backbone(arch, bb_dropout=0.1, n_layers=None):
    if ARCHS[arch]["family"] == FAMILY:
        bb, hid, causal, ctx, fam = AL.load_backbone(ARCHS[arch]["path"])
        with torch.no_grad(): dim = bb(input_ids=torch.zeros(1, 4, dtype=torch.long), attention_mask=torch.ones(1, 4, dtype=torch.long)).last_hidden_state.shape[-1]
        return bb, dim, causal, ctx, FAMILY
    return _orig_build_backbone(arch, bb_dropout, n_layers)
print("registered architectures:", {k: v["path"] for k, v in ARCHS.items()})


In [ ]:

PLAN = {"E7_modernbert_full": dict(method="full", ctx=True, arch=FAMILY)}
run_experiment("E7_modernbert_full", epochs=3, patience=2, **PLAN["E7_modernbert_full"])      # full fine-tuning of the Phase 1 ModernBERT-style encoder, both domains
BEST = "E7_modernbert_full"


In [ ]:
# ================= CELL 15 — post-hoc improvements with NO retraining: ensembling, transition smoothing, kNN interpolation =================
def load_arr(name, dom, nm): return np.load(WORK / "runs" / f"{name}_{dom}" / f"{nm}.npy")
def ens_probs(names, dom, split):
    lp = [np.log(softmax_np(load_arr(n, dom, f"{split}_logits")) + 1e-9) for n in names]; return softmax_np(np.mean(lp, 0))
def posthoc_probs(names, dom, split, smooth=0.0, knn_lam=1.0, k=10, tau=0.1):
    p = ens_probs(names, dom, split)
    if knn_lam < 1.0:   # neighbours from train embeddings of the FIRST run in `names`
        p = knn_interpolate(load_arr(names[0], dom, f"{split}_emb").astype(np.float32), load_arr(names[0], dom, "train_emb").astype(np.float32),
                            load_arr(names[0], dom, "train_y"), p, K, k=k, tau=tau, lam=knn_lam)
    return p
def predict_labels(probs, exs, logT, logS, smooth): return smooth_split(probs, exs, logT, logS, smooth) if smooth > 0 else probs.argmax(1)

def tune_posthoc(names, dom):
    ex_dv = build_examples(DOCS[dom]["dev"], True); y = np.array([e["y"] for e in ex_dv])
    logT, logS = transition_logprobs(DOCS[dom]["train"], LABEL2ID, K); rows = []
    for knn_lam in (1.0, 0.8, 0.6):
        p = posthoc_probs(names, dom, "dev", knn_lam=knn_lam)
        for sm in (0.0, 0.25, 0.5, 1.0):
            rows.append(dict(domain=dom, knn_lam=knn_lam, smooth=sm, dev_macro_f1=prf_report(y, predict_labels(p, ex_dv, logT, logS, sm), K)["macro_f1"]))
    t = pd.DataFrame(rows); base = t[(t.knn_lam == 1.0) & (t.smooth == 0.0)].dev_macro_f1.iloc[0]
    print(t.round(4).to_string(index=False)); print(f"baseline {base:.4f}; best {t.dev_macro_f1.max():.4f}"); return t, base

CANDIDATES = {dom: [BEST] for dom in ("CL", "IT")}      # list one or more finished run names per domain (an ensemble if several)
for dom in ("CL", "IT"): _ = tune_posthoc(CANDIDATES[dom], dom)
print("\nRule: adopt a post-hoc setting only if it gains >= 0.01 on BOTH domains; dev has very few documents and small gains are noise.")


In [ ]:
# ================= CELL 16 — FINAL PREDICTIONS + VALIDATION + ZIP =================
# The observed test format is {document_id: [sentence, ...]}.
# Build a separate predictions object: {document_id: [predicted_label, ...]}.
FINAL = {
    "CL": dict(names=[BEST], knn_lam=1.0, smooth=0.0),
    "IT": dict(names=[BEST], knn_lam=1.0, smooth=0.0),
}

PREDICTIONS = {}
for dom in ("CL", "IT"):
    f = FINAL[dom]
    ex_te = build_examples(DOCS[dom]["test"], ctx=PLAN[f["names"][0]]["ctx"])
    probs = posthoc_probs(f["names"], dom, "test", knn_lam=f["knn_lam"])
    logT, logS = transition_logprobs(
        DOCS[dom]["train"] + DOCS[dom]["dev"], LABEL2ID, K
    )
    yhat = predict_labels(probs, ex_te, logT, logS, f["smooth"])

    by_doc = defaultdict(list)
    for e, y_ in sorted(zip(ex_te, yhat), key=lambda pair: (pair[0]["doc"], pair[0]["i"])):
        by_doc[e["doc"]].append(LABELS[int(y_)])

    pred_obj = {}
    for d in DOCS[dom]["test"]:
        labs = by_doc[d["id"]]
        assert len(labs) == len(d["sents"]), (
            f"{dom}/{d['id']}: predicted {len(labs)} labels for {len(d['sents'])} sentences"
        )
        assert all(lab in LABEL2ID for lab in labs)
        pred_obj[d["id"]] = labs

    # Validate exact IDs/order and sentence-level label counts against the input test set.
    assert list(pred_obj.keys()) == list(TEST_OBJ[dom].keys()), f"{dom}: document IDs/order mismatch"
    for doc_id, sentences in TEST_OBJ[dom].items():
        assert len(pred_obj[doc_id]) == len(sentences), f"{dom}/{doc_id}: label count mismatch"

    PREDICTIONS[dom] = pred_obj
    out_file = WORK / f"prediction_{dom}.json"
    with open(out_file, "w", encoding="utf-8") as f_out:
        json.dump(pred_obj, f_out, ensure_ascii=False, indent=2)

    print(f"{dom}: {sum(map(len, pred_obj.values()))} sentence labels")
    print("  label distribution:", dict(Counter(l for labs in pred_obj.values() for l in labs)))
    print("  saved:", out_file)

# Combined keyed output; check the capstone submission instructions for the required final filename/schema.
combined = {"CL": PREDICTIONS["CL"], "IT": PREDICTIONS["IT"]}
combined_file = WORK / "prediction.json"
with open(combined_file, "w", encoding="utf-8") as f_out:
    json.dump(combined, f_out, ensure_ascii=False, indent=2)

zip_path = Path("/kaggle/working/submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(combined_file, "prediction.json")
    zf.write(WORK / "prediction_CL.json", "prediction_CL.json")
    zf.write(WORK / "prediction_IT.json", "prediction_IT.json")

with zipfile.ZipFile(zip_path) as zf:
    print("ZIP files:", zf.namelist())
    check = json.load(zf.open("prediction.json"))
    assert list(check.keys()) == ["CL", "IT"]
    for dom in ("CL", "IT"):
        assert list(check[dom].keys()) == list(TEST_OBJ[dom].keys())
        for doc_id, sents in TEST_OBJ[dom].items():
            assert len(check[dom][doc_id]) == len(sents)

print("Prediction files written and structural validation passed.")


# ================= DURABLE RECOVERY BUNDLE =================
# This creates a single archive containing resumable latest.pt checkpoints,
# best.pt inference checkpoints, experiment outputs, ledger, and predictions.
# Download this ZIP or publish it as a dedicated Kaggle Dataset before ending the session.
RECOVERY_DIR = Path("/kaggle/working/phase3_recovery")
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)
RECOVERY_ZIP = Path("/kaggle/working/phase3_recovery.zip")
manifest = {
    "created_unix": time.time(),
    "project": "CS781 Phase 3 Rhetorical Role Prediction",
    "source_dir": str(WORK),
    "checkpoint_policy": "latest.pt saved after every completed epoch; best.pt saved for best dev macro-F1",
    "contains": [
        "experiment latest.pt and best.pt checkpoints where available",
        "result.json, logits, embeddings, and train_y.npy for completed experiments",
        "ledger.csv",
        "prediction.json and per-domain prediction files when generated",
    ],
    "durability_note": (
        "The ZIP is in /kaggle/working and is not an external durable backup until downloaded "
        "or uploaded as a version of a dedicated Kaggle Dataset."
    ),
}
with open(RECOVERY_DIR / "recovery_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

if RECOVERY_ZIP.exists():
    RECOVERY_ZIP.unlink()
with zipfile.ZipFile(RECOVERY_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # Include phase3_rr artifacts without recursively including the recovery ZIP itself.
    for src in WORK.rglob("*"):
        if src.is_file() and not src.name.endswith(".tmp"):
            zf.write(src, Path("phase3_rr") / src.relative_to(WORK))
    zf.write(RECOVERY_DIR / "recovery_manifest.json", "recovery_manifest.json")

with zipfile.ZipFile(RECOVERY_ZIP, "r") as zf:
    bad = zf.testzip()
    assert bad is None, f"Recovery ZIP failed validation at {bad}"
    print("Recovery bundle verified.")
    print("Files included:", len(zf.namelist()))
print("Recovery ZIP:", RECOVERY_ZIP)
print("Size (MB):", round(RECOVERY_ZIP.stat().st_size / (1024**2), 2))
print("The final cell below will also publish the recovery checkpoints to the dedicated Kaggle Dataset.")


# ================= FINAL DURABLE DATASET PUBLISH =================
# Publishes latest/best checkpoints, results, ledger, and predictions as a new
# version of the dedicated Kaggle Dataset. This is separate from the local ZIP.
if DURABLE_PUBLISH_ENABLED:
    publish_recovery_dataset(reason="final_notebook_recovery")
else:
    print("FINAL DURABLE PUBLISH SKIPPED: configure Kaggle Secrets in Cell 0b.")
    print("Local recovery ZIP remains at:", RECOVERY_ZIP)
